In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('../data/train.csv')

df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))

df.to_csv('../data/train.csv', index=False)




In [ ]:
print(df.isnull().sum())

df['Embarked'] = df['Embarked'].fillna('S')

df['HasCabin'] = df['Cabin'].notna().astype(int)
#Cabini bilgisi hakkında sütun oluşturur ve kabin bilgisi varsa 1, yoksa 0 olarak işaretle
df.drop('Cabin', axis=1, inplace=True)

print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
HasCabin       0
dtype: int64


In [24]:
print(df.isna().sum()) #temizlik bitti.

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
HasCabin       0
dtype: int64


In [25]:

df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.')

df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')


df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print(df['Title'].value_counts())
print(df[['FamilySize', 'IsAlone']].head(10))


Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64
   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0


<>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\ashe\AppData\Local\Temp\ipykernel_41072\2143953523.py:1: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.')


In [26]:
from sklearn.preprocessing import LabelEncoder

df.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])
df['Title'] = le.fit_transform(df['Title'])

print(df.head())
print(df.dtypes)

   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  HasCabin  \
0         0       3    1  22.0      1      0   7.2500         2         0   
1         1       1    0  38.0      1      0  71.2833         0         1   
2         1       3    0  26.0      0      0   7.9250         2         0   
3         1       1    0  35.0      1      0  53.1000         2         1   
4         0       3    1  35.0      0      0   8.0500         2         0   

   Title  FamilySize  IsAlone  
0      2           2        0  
1      3           2        0  
2      1           1        1  
3      3           2        0  
4      2           1        1  
Survived        int64
Pclass          int64
Sex             int64
Age           float64
SibSp           int64
Parch           int64
Fare          float64
Embarked        int64
HasCabin        int64
Title           int64
FamilySize      int64
IsAlone         int64
dtype: object


In [27]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df.drop('Survived', axis=1)
y = df['Survived']


#egitim 80 test 20.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500)
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5)
    print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}")



Random Forest: 0.799 ± 0.034
Gradient Boosting: 0.826 ± 0.043
Logistic Regression: 0.803 ± 0.027


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2]
}

gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid, cv=5, scoring='accuracy', n_jobs=-1)

gs.fit(X_train, y_train)

print("En iyi parametrelr:", gs.best_params_)
print("En iyi skor:", f"{gs.best_score_:.3f}")



En iyi parametreler: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
En iyi skor: 0.826


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_model = gs.best_estimator_
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred),"")
print("Classification report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[96 14]
 [23 46]] 
Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       110
           1       0.77      0.67      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.78       179
weighted avg       0.79      0.79      0.79       179



In [ ]:
test = pd.read_csv('../data/test.csv')

print(test.isnull().sum())

#Ttemizlik işlemleri
test['Age'] = test.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
test['Fare'] = test['Fare'].fillna(test['Fare'].median())
test['HasCabin'] = test['Cabin'].notna().astype(int)
test.drop('Cabin', axis=1, inplace=True)

print(test.isnull().sum())
test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\. ')
test['Title'] = test['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
test['Title'] = test['Title'].replace(['Mlle', 'Ms'], 'Miss')
test['Title'] = test['Title'].replace('Mme', 'Mrs')
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

passenger_ids = test['PassengerId']  # Tahmin sonuçlarını kaydetmek için PassengerId'leri saklıyoruz
test.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
test['Sex'] = le.fit_transform(test['Sex'])
test['Embarked'] = le.fit_transform(test['Embarked'])
test['Title'] = le.fit_transform(test['Title'])

predictions = best_model.predict(test)

submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions
})
submission.to_csv('../data/submission.csv', index=False)
print(submission.head(10))
print("The end.")



PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64
PassengerId    0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
HasCabin       0
dtype: int64
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0
The end.


<>:12: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:12: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
C:\Users\ashe\AppData\Local\Temp\ipykernel_41072\1939861719.py:12: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\. ')


In [ ]:

df['FarePerPerson'] = df['Fare'] / df['FamilySize']

df['Age_Pclass'] = df['Age'] * df['Pclass']

df['FamilyGroup'] = pd.cut(df['FamilySize'], 
    bins=[0, 1, 4, 20], 
    labels=['alone', 'small', 'large'])
df['FamilyGroup'] = LabelEncoder().fit_transform(df['FamilyGroup'])

print(df[['FarePerPerson', 'Age_Pclass', 'FamilyGroup']].head())

   FarePerPerson  Age_Pclass  FamilyGroup
0        3.62500        66.0            2
1       35.64165        38.0            2
2        7.92500        78.0            0
3       26.55000        35.0            2
4        8.05000       105.0            0


In [ ]:
from sklearn.ensemble import VotingClassifier

model1 = GradientBoostingClassifier(learning_rate=0.1, max_depth=3, 
                                     n_estimators=100, random_state=42)
model2 = RandomForestClassifier(n_estimators=200, random_state=42)
model3 = LogisticRegression(max_iter=500)

ensemble = VotingClassifier(
    estimators=[('gb', model1), ('rf', model2), ('lr', model3)],
    voting='soft'  
    # olasılıkları birleştirmek
)

scores = cross_val_score(ensemble, X_train, y_train, cv=5)
print(f"Ensemble: {scores.mean():.3f} ± {scores.std():.3f}")

Ensemble: 0.823 ± 0.025


In [ ]:
# Yeni featurelare
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
df['Age_Pclass']    = df['Age'] * df['Pclass']
df['FamilyGroup']   = pd.cut(df['FamilySize'], 
                              bins=[0, 1, 4, 20], 
                              labels=['alone', 'small', 'large'])
df['FamilyGroup'] = LabelEncoder().fit_transform(df['FamilyGroup'])

# X ve y'yi güncelle
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scores = cross_val_score(
    GradientBoostingClassifier(learning_rate=0.1, max_depth=3, 
                                n_estimators=100, random_state=42),
    X_train, y_train, cv=5)
print(f"Gradient Boosting (neww features,): {scores.mean():.3f} ± {scores.std():.3f}")

scores2 = cross_val_score(ensemble, X_train, y_train, cv=5)
print(f"Ensemble (new features): {scores2.mean():.3f} ± {scores2.std():.3f}")

ensemble = VotingClassifier(
    estimators=[('gb', model1), ('rf', model2)],
    voting='soft'
)

model3 = LogisticRegression(max_iter=2000)

Gradient Boosting (yeni features): 0.825 ± 0.038
Ensemble (yeni features): 0.827 ± 0.033


In [ ]:
test['FarePerPerson'] = test['Fare'] / test['FamilySize']
test['Age_Pclass']    = test['Age'] * test['Pclass']
test['FamilyGroup']   = pd.cut(test['FamilySize'], 
                                bins=[0, 1, 4, 20], 
                                labels=['alone', 'small', 'large'])
test['FamilyGroup'] = LabelEncoder().fit_transform(test['FamilyGroup'])

ensemble.fit(X_train, y_train)
predictions = ensemble.predict(test)
# Yeni submission dosyası
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': predictions
})
submission.to_csv('../data/submission_v2.csv', index=False)
print("submission_v2.csv oluştu!")

submission_v2.csv oluştu!
